# RAG Step 3 — Retrieval + Generation

**Where we are:**
```
Step 1 (done) → statements/   raw data
Step 2 (done) → ChromaDB      vectors + metadata
Step 3 (this) → Retriever     query → top-k chunks
              → Chain         chunks + query → GPT-4 → answer
Step 4 (next) → Chat UI       Streamlit chat interface
```

**Four concepts we'll decode:**
1. Retrieval — finding the right chunks for a query
2. Prompt assembly — what the LLM actually sees
3. Generation — the LLM call and why temperature matters
4. Multi-turn — how conversation history works

In [1]:
import os, sys
from pathlib import Path
from dotenv import load_dotenv
import chromadb
from openai import OpenAI

load_dotenv()

# Add project root to path so we can import from rag/
sys.path.insert(0, str(Path("..").resolve()))

CHROMA_DIR = Path("..") / "data" / "chroma"

openai_client  = OpenAI()
chroma_client  = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection     = chroma_client.get_collection("cardwise")

print(f"Collection has {collection.count()} vectors")

Collection has 118 vectors


---
## Concept 1: Retrieval

**The problem:** GPT-4 doesn't know Alice's spending data or the exact card terms. We need to inject that knowledge.

**The solution:** Before calling the LLM, we find the chunks most relevant to the user's question and hand them to the LLM as part of the prompt.

**The mechanism:**
1. Embed the query with the same model used at index time (`text-embedding-ada-002`)
2. Run cosine similarity search in ChromaDB
3. Filter by `user_id` so we only get Alice's data

We run **two separate searches** per query:
- Search A → Alice's spend chunks (what actually happened)
- Search B → Card rule chunks (what the card promises)

Without both, the LLM can see spending but not compare it to what *could* have been earned.

In [2]:
query   = "Am I getting good value from dining spend on my cards?"
user_id = "alice"

# Step 1: embed the query
query_vec = openai_client.embeddings.create(
    model="text-embedding-ada-002",
    input=[query],
).data[0].embedding

print(f"Query embedded → {len(query_vec)}-dim vector")

Query embedded → 1536-dim vector


In [3]:
# Step 2a: find Alice's most relevant spend chunks
spend_results = collection.query(
    query_embeddings=[query_vec],
    n_results=6,
    where={"$and": [{"user_id": user_id}, {"chunk_type": "spend_summary"}]},
    include=["documents", "distances"],
)

spend_chunks = spend_results["documents"][0]
distances    = spend_results["distances"][0]

print("TOP SPEND CHUNKS:")
for chunk, dist in zip(spend_chunks, distances):
    print(f"  [{dist:.3f}] {chunk}")

TOP SPEND CHUNKS:
  [0.167] Alice Chen (alice) spent $10.03 on dining using chase-sapphire-preferred in 2026-04 across 1 transaction(s) and earned 30 points at 3x (≈$0.38).
  [0.169] Alice Chen (alice) spent $36.25 on dining using chase-sapphire-preferred in 2026-01 across 1 transaction(s) and earned 109 points at 3x (≈$1.36).
  [0.169] Alice Chen (alice) spent $205.91 on dining using citi-double-cash in 2026-04 across 8 transaction(s) and earned $4.11 cashback at 2%.
  [0.170] Alice Chen (alice) spent $43.14 on dining using chase-sapphire-preferred in 2026-05 across 2 transaction(s) and earned 129 points at 3x (≈$1.61).
  [0.170] Alice Chen (alice) spent $293.71 on dining using citi-double-cash in 2025-12 across 11 transaction(s) and earned $5.87 cashback at 2%.
  [0.170] Alice Chen (alice) spent $160.58 on dining using chase-sapphire-preferred in 2026-03 across 2 transaction(s) and earned 482 points at 3x (≈$6.03).


In [4]:
# Step 2b: find relevant card rule chunks (no user filter — global facts)
rule_results = collection.query(
    query_embeddings=[query_vec],
    n_results=2,
    where={"chunk_type": "card_rules"},
    include=["documents", "distances"],
)

rule_chunks = rule_results["documents"][0]

print("TOP CARD RULE CHUNKS:")
for chunk, dist in zip(rule_chunks, rule_results["distances"][0]):
    print(f"  [{dist:.3f}] {chunk[:120]}...")

TOP CARD RULE CHUNKS:
  [0.182] American Express Gold (annual fee: $250). Earns Membership Rewards points, valued at 1.0¢ per point.   4x Membership Rew...
  [0.206] Chase Sapphire Preferred (annual fee: $95). Earns Ultimate Rewards points, valued at 1.25¢ per point.   3x Ultimate Rewa...


**Notice:** distances are lower for more relevant chunks. The spend chunks that surfaced are specifically dining-related, even though our query didn't use those exact words — that's semantic search at work.

The card rule chunk that surfaced should be the one with the best dining rate — let's verify.

In [5]:
# What card rules surfaced? Print them in full.
print("CARD RULES RETRIEVED:")
for i, chunk in enumerate(rule_chunks, 1):
    print(f"\n[{i}] {chunk}")

CARD RULES RETRIEVED:

[1] American Express Gold (annual fee: $250). Earns Membership Rewards points, valued at 1.0¢ per point.   4x Membership Rewards on dining.   4x Membership Rewards on groceries.   3x Membership Rewards on travel.   1x Membership Rewards on other. Key perks: $120 dining credit (Grubhub, Cheesecake Factory, and more); $120 Uber Cash annually; No foreign transaction fees.

[2] Chase Sapphire Preferred (annual fee: $95). Earns Ultimate Rewards points, valued at 1.25¢ per point.   3x Ultimate Rewards on travel.   3x Ultimate Rewards on dining.   1x Ultimate Rewards on other.   3x on streaming: Netflix, Spotify, Hulu, Disney+, Apple TV+, HBO Max. Key perks: No foreign transaction fees; Primary car rental collision insurance; Trip cancellation & interruption insurance.


---
## Concept 2: Prompt Assembly

**The 'Augmented' in RAG.** Before the LLM sees the user's question, we inject the retrieved chunks as context. This is the core trick.

The LLM receives three things:

```
1. System prompt  — role definition + rules (no hallucinating card terms)
2. Context block  — retrieved spend summaries + card rules
3. User question  — the actual query
```

The **system prompt** shapes behavior. The **context block** is what makes the answer grounded. Without it, GPT-4 would invent card terms or give generic advice.

Let's see the exact prompt that goes to the API.

In [6]:
SYSTEM_PROMPT = """You are a personal credit card advisor with access to the user's actual spend history and their card reward terms.

Rules:
- Answer using only the provided context. Never invent card terms.
- Be specific: quote dollar amounts, reward rates, and point values from the context.
- If the user is missing a better reward rate on another card they hold, call it out clearly.
- Keep answers concise — 3 to 5 sentences unless a breakdown is genuinely useful."""


def build_context(spend_chunks, rule_chunks):
    parts = []
    if spend_chunks:
        parts.append("USER SPEND HISTORY:\n" + "\n".join(f"- {c}" for c in spend_chunks))
    if rule_chunks:
        parts.append("CARD REWARD TERMS:\n" + "\n".join(f"- {c}" for c in rule_chunks))
    return "\n\n".join(parts)


context = build_context(spend_chunks, rule_chunks)

# This is exactly what GPT-4 sees in the user message
user_message = f"Context:\n{context}\n\nQuestion: {query}"

print("=== SYSTEM PROMPT ===")
print(SYSTEM_PROMPT)
print()
print("=== USER MESSAGE (first 800 chars) ===")
print(user_message[:800], "...")

=== SYSTEM PROMPT ===
You are a personal credit card advisor with access to the user's actual spend history and their card reward terms.

Rules:
- Answer using only the provided context. Never invent card terms.
- Be specific: quote dollar amounts, reward rates, and point values from the context.
- If the user is missing a better reward rate on another card they hold, call it out clearly.
- Keep answers concise — 3 to 5 sentences unless a breakdown is genuinely useful.

=== USER MESSAGE (first 800 chars) ===
Context:
USER SPEND HISTORY:
- Alice Chen (alice) spent $10.03 on dining using chase-sapphire-preferred in 2026-04 across 1 transaction(s) and earned 30 points at 3x (≈$0.38).
- Alice Chen (alice) spent $36.25 on dining using chase-sapphire-preferred in 2026-01 across 1 transaction(s) and earned 109 points at 3x (≈$1.36).
- Alice Chen (alice) spent $205.91 on dining using citi-double-cash in 2026-04 across 8 transaction(s) and earned $4.11 cashback at 2%.
- Alice Chen (alice) spent

**Key observation:** the LLM doesn't do any retrieval itself. It only sees what we hand it. If a chunk isn't retrieved, the LLM has no way to reference it. Retrieval quality directly determines answer quality — this is why chunking strategy matters so much.

The context structure (`USER SPEND HISTORY:` / `CARD REWARD TERMS:`) helps the LLM distinguish "what happened" from "what the card says" — two different types of evidence it needs to reason about.

---
## Concept 3: Generation

Now we send the assembled prompt to GPT-4 and get a grounded answer.

**Temperature** controls randomness:
- `0.0` → completely deterministic, same answer every time
- `1.0` → creative/unpredictable
- `0.3` → factual but naturally worded (right for advisory responses)

**Model choice: `gpt-4o`** — faster and cheaper than `gpt-4`, same quality for this task.

In [7]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": user_message},
]

response = openai_client.chat.completions.create(
    model="gpt-4o",
    messages=messages,
    temperature=0.3,
)

answer = response.choices[0].message.content

print("ANSWER:")
print(answer)
print()
print(f"Tokens used — prompt: {response.usage.prompt_tokens}, completion: {response.usage.completion_tokens}")

ANSWER:
You're currently using the Chase Sapphire Preferred for most of your dining spend, earning 3x points, which are valued at 1.25¢ per point. This gives you an effective return of 3.75% on dining. However, you also have the American Express Gold card, which offers 4x points on dining, valued at 1¢ per point, providing a 4% return. To maximize your rewards, consider using the American Express Gold card for dining purchases to take advantage of the higher reward rate.

Tokens used — prompt: 590, completion: 106


**The answer is grounded** — every dollar figure and rate it mentions came from the context we injected. The LLM didn't know Alice's data or card terms before we retrieved them.

Notice the token count: the prompt tokens are much higher than the completion tokens. That's normal for RAG — most of the cost is injecting context, not generating the answer. This is why chunk design matters for cost too.

---
## Concept 4: Multi-turn — Conversation History

**The problem:** each API call is stateless. If you ask a follow-up question, GPT-4 has no memory of the previous exchange.

**The solution:** pass prior turns as messages. OpenAI's chat API accepts a list of `{role, content}` pairs. You simply append each turn and send the whole list.

```
Turn 1:  [system] + [user: Q1] → [assistant: A1]
Turn 2:  [system] + [user: Q1] + [assistant: A1] + [user: Q2] → [assistant: A2]
Turn 3:  [system] + [user: Q1] + [assistant: A1] + [user: Q2] + [assistant: A2] + [user: Q3] → ...
```

The context retrieval still runs fresh on each turn — we retrieve chunks relevant to the **current** question, not the whole conversation. History handles the "you just said X" part; retrieval handles the "what does my data say" part.

In [8]:
def ask(query, user_id, history=None):
    """Single function that does retrieve → assemble → generate."""

    # 1. Retrieve relevant chunks for THIS query
    q_vec = openai_client.embeddings.create(
        model="text-embedding-ada-002", input=[query]
    ).data[0].embedding

    spend = collection.query(
        query_embeddings=[q_vec], n_results=6,
        where={"$and": [{"user_id": user_id}, {"chunk_type": "spend_summary"}]},
        include=["documents"],
    )["documents"][0]

    rules = collection.query(
        query_embeddings=[q_vec], n_results=2,
        where={"chunk_type": "card_rules"},
        include=["documents"],
    )["documents"][0]

    # 2. Assemble context
    context = build_context(spend, rules)

    # 3. Build message list — system + history + current question with context
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    if history:
        messages.extend(history)   # prior turns go here
    messages.append({"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"})

    # 4. Generate
    response = openai_client.chat.completions.create(
        model="gpt-4o", messages=messages, temperature=0.3
    )
    return response.choices[0].message.content


# --- Turn 1 ---
q1 = "Which category am I spending the most on and how much am I earning from it?"
a1 = ask(q1, user_id="alice")
print(f"Q1: {q1}")
print(f"A1: {a1}")

Q1: Which category am I spending the most on and how much am I earning from it?
A1: You are spending the most on groceries, with a total spend of $1,460.70 across three transactions in 2026. Using the Citi Double Cash card, you earned $29.22 in cashback at a 2% rate. However, during 2026-Q1, you could have earned 5% cashback on groceries with your Discover it Cash Back card, which would have resulted in $73.04 cashback for the same spend, assuming you stayed within the $1,500 quarterly cap.


In [9]:
# Build history from Turn 1 — these are the raw OpenAI message dicts
history = [
    {"role": "user",      "content": q1},
    {"role": "assistant", "content": a1},
]

# --- Turn 2 (follow-up — no need to repeat Q1 context) ---
q2 = "What should I do differently next month to get more value?"
a2 = ask(q2, user_id="alice", history=history)
print(f"Q2: {q2}")
print(f"A2: {a2}")

Q2: What should I do differently next month to get more value?
A2: To maximize your rewards next month, consider using your American Express Gold card for grocery purchases. It offers 4x Membership Rewards points on groceries, which is significantly better than the 2% cashback you are currently earning with the Citi Double Cash card. For example, if you spend $500 on groceries, you would earn 2,000 Membership Rewards points, valued at $20, compared to just $10 cashback with your current card. This switch would effectively double your rewards value for grocery spending.


In [10]:
# --- Turn 3 (follow-up on Turn 2) ---
history += [
    {"role": "user",      "content": q2},
    {"role": "assistant", "content": a2},
]

q3 = "Should I book a concert ticket this month to maximize rewards?"
a3 = ask(q3, user_id="alice", history=history)
print(f"Q3: {q3}")
print(f"A3: {a3}")

Q3: Should I book a concert ticket this month to maximize rewards?
A3: If you plan to book a concert ticket this month, you should continue using your Citi Double Cash card, as it offers 2% cashback on all purchases, including entertainment. However, if the concert ticket purchase falls under a streaming service or platform that qualifies for your Chase Sapphire Preferred card's 3x streaming category, consider using that card instead. Otherwise, for general entertainment purchases, the Citi Double Cash card remains your best option for maximizing rewards.


---
## Summary — what the full chain looks like

```
User query
    │
    ▼
Embed query (text-embedding-ada-002)
    │
    ├─ ChromaDB search (user spend, top 6)
    └─ ChromaDB search (card rules, top 2)
    │
    ▼
Assemble prompt
  [system prompt]
  [conversation history]   ← grows with each turn
  [user: context + question]
    │
    ▼
GPT-4o (temp=0.3)
    │
    ▼
Answer (grounded in retrieved context)
```

**What makes this RAG and not just "calling GPT-4":**
- The LLM has no prior knowledge of Alice's data or exact card terms
- Every factual claim in the answer traces back to a retrieved chunk
- The system prompt explicitly prohibits inventing card terms
- Retrieval is semantic (meaning-based), not keyword-based

**Next:** wrap this in a Streamlit chat UI (`components/chat.py`) so users can type queries and see answers in the app.